# Capstone: the professor's GRU and CNN-GRU as habit models - 4w

"Will this appliance be switched on in the next hour?", the same question, split and scoring as the
random forest, on 123 REFIT and 54 UK-DALE tasks; run 2 (--weeks 4 --epochs 60), fixed before seeing its results: the networks get the same 4 weeks of history as the forest's features, and the GRU is not cut off by the epoch cap.

**Settings:** Accelerator = GPU, dataset `capstone-bundle` (latest version) under Input, Internet = On
(only for the phone notifications). Save Version > Save & Run All (Commit). Progress streams to the
log, one line per task.

**Phone notifications (optional):** add a Kaggle secret `NTFY_URL` = `https://ntfy.sh/<your topic>`
(Add-ons > Secrets, attached to this notebook); if Kaggle keeps dropping it, uncomment the first
`os.environ['NTFY_URL'] = ...` line instead. You get: started, progress every 30 min, each step
done or failed, finished.

**A new notebook copies none of these settings:** add the input, GPU, Internet (and the topic) each time.

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: check the Accelerator setting'
notify('Habit networks (4w): started', f'GPU: {gpu}\nREFIT 123 tasks, then UK-DALE 54; x2 networks --weeks 4 --epochs 60',
       'default' if torch.cuda.is_available() else 'high', 'rocket')

In [ ]:
ok = run_step('Tests', 'python -m pytest -q tests/test_habit_networks.py', 'logs/tests.log', summary_lines=2)
ok &= run_step('REFIT Habit networks (4w)', 'python -u scripts/eval_habit_networks.py --weeks 4 --epochs 60', 'logs/refit_habit_networks_4w.log')

In [ ]:
ok &= run_step('UK-DALE Habit networks (4w)', 'python -u scripts/eval_habit_networks.py --weeks 4 --epochs 60', 'logs/ukdale_habit_networks_4w.log', env={'CAPSTONE_DATASET': 'ukdale'})

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/habit_networks_4w.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ['data/results/habit_networks_4w.csv', 'data/results/ukdale/habit_networks_4w.csv',
              'logs/refit_habit_networks_4w.log', 'logs/ukdale_habit_networks_4w.log']:
        if pathlib.Path(f).exists():
            zf.write(f, f)
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
print(sorted(os.listdir('/kaggle/working')))
notify('Habit networks (4w): FINISHED' if ok else 'Habit networks (4w): finished WITH ERRORS',
       'habit_networks_4w.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')